# HW2 — Profile & Optimize an Autoregressive Decode Loop

**Lecture mapping:** L1 §07 (Profiling) · L2 §01 (prefill/decode, KV cache) · L2 §03 (engine optimizations)

The harness below defines a deliberately slow greedy-decode loop (**V0**): no KV
cache (it recomputes the whole sequence every step), fp32 eager, and a host sync
every step. Profile it, find the bottlenecks, and write a fast, numerically
identical replacement.

## What you implement

| Part | Function |
|------|----------|
| 1 | `profile` — wrap a loop in `torch.profiler`, print a table, export a Chrome trace |
| 2 | `optimized_loop` — fast greedy decode, same tokens as V0 (fp32) |
| 3 | `generate_optimized` — build, warm up, and time your optimized generation |
| 4 | Writeup Q1–Q4 |

## Speedup targets

`optimized_loop` end-to-end vs the V0 baseline:

| Speedup vs V0 | |
|---------------|--|
| ≥ 4.0× | excellent |
| ≥ 3.0× | good |
| ≥ 1.5× | some speedup |
| < 1.5× | little to no speedup |

The speedup only counts if `optimized_loop` passes the fp32 correctness check and
the timed run is on a GPU.

## Rules

- PyTorch + `requirements.txt` only — no vLLM / TensorRT-LLM / SGLang.
- Don't edit the harness cell.
- `optimized_loop` must reproduce the baseline's greedy tokens exactly on the same
  fp32 model. `generate_optimized` may switch to bf16 for the timed run.

## Where to look

- **KV cache** (L2 §01) — the baseline throws it away every step. Biggest win.
- **Host syncs** (L1 §07) — `.item()` on the critical path serializes CPU↔GPU.
- **`torch.compile` / CUDA graphs** (L2 §03) — fuse ops, cut launch overhead.
- **dtype** — bf16 for the timed run (small on a model this size).

## Cell types

- **DO NOT EDIT** — fixed harness.
- **YOUR IMPLEMENTATION** — replace `raise NotImplementedError`.
- **SELF-CHECK** — asserts that must pass.
- **WRITEUP** — answer in the markdown cell.

Run top-to-bottom on the GPU. Submit the executed notebook plus the files under
`results/`.

## The fixed harness (DO NOT EDIT)

Tiny 2-layer Llama, the V0 baseline, the correctness check, the timing helper,
and the speedup targets.

In [1]:
# DO NOT EDIT — editing this cell invalidates your speedup numbers.
import os, time

import torch
from transformers import LlamaConfig, LlamaForCausalLM

RESULTS_DIR = os.path.join("results", "hw2")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 0
PROMPT_LEN = 64
MAX_NEW_TOKENS = 128

# Speedup targets (optimized end-to-end vs V0 baseline).
TIERS = [
    (4.0, ">= 4.0x  (excellent)"),
    (3.0, ">= 3.0x  (good)"),
    (1.5, ">= 1.5x  (some speedup)"),
    (0.0, "< 1.5x  (little to no speedup)"),
]


def device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def _config() -> LlamaConfig:
    # Small enough to iterate fast, real enough to profile meaningfully.
    return LlamaConfig(
        vocab_size=32000,
        hidden_size=512,
        intermediate_size=1376,
        num_hidden_layers=2,
        num_attention_heads=8,
        num_key_value_heads=8,
        max_position_embeddings=4096,
    )


def build_model_and_input(dtype: torch.dtype = torch.float32):
    """Return (model, input_ids) on the active device with fixed random weights.

    The same SEED is used every call, so the V0 baseline and your optimized run
    see identical weights and the same prompt.
    """
    torch.manual_seed(SEED)
    model = LlamaForCausalLM(_config()).to(device=device(), dtype=dtype).eval()
    input_ids = torch.randint(0, 32000, (1, PROMPT_LEN), device=device())
    return model, input_ids


@torch.no_grad()
def baseline_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """V0 — intentionally slow. Greedy decode with NO KV cache: every step
    re-runs the model over the whole growing sequence (O(n^2)), and forces a
    host sync each step (the `.item()`) — exactly the anti-patterns from lecture.

    Returns the generated token ids, shape (1, max_new_tokens).
    """
    generated = input_ids
    out_tokens = []
    for _ in range(max_new_tokens):
        out = model(input_ids=generated, use_cache=False)
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        _ = next_tok.item()                       # forced host sync every step
        generated = torch.cat([generated, next_tok], dim=1)
        out_tokens.append(next_tok)
    return torch.cat(out_tokens, dim=1)


def time_loop(loop_fn, *args, n_warmup: int = 1, n_iters: int = 3) -> float:
    """Average seconds per full generation of `loop_fn(*args)`."""
    for _ in range(n_warmup):
        loop_fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(n_iters):
            loop_fn(*args)
        end.record()
        torch.cuda.synchronize()
        return (start.elapsed_time(end) / 1e3) / n_iters
    t0 = time.perf_counter()
    for _ in range(n_iters):
        loop_fn(*args)
    return (time.perf_counter() - t0) / n_iters


def check_correctness(reference: torch.Tensor, candidate: torch.Tensor) -> bool:
    """Greedy decoding is deterministic, so a correct optimized loop must
    reproduce the baseline's token ids EXACTLY when given the same fp32 model."""
    return (reference.shape == candidate.shape
            and torch.equal(reference.cpu(), candidate.cpu()))


def speedup_tier(speedup: float) -> str:
    for threshold, label in TIERS:
        if speedup >= threshold:
            return label
    return TIERS[-1][1]


print(f"device={device()}  prompt_len={PROMPT_LEN}  new_tokens={MAX_NEW_TOKENS}")

device=cuda  prompt_len=64  new_tokens=128


## Part 1 — profile a generation loop

See the L1 §07 `torch.profiler` snippet. You will use this on both the baseline
and your optimized loop, and read the traces at
[ui.perfetto.dev](https://ui.perfetto.dev).

In [2]:
from torch import profiler
def profile(loop_fn, model, input_ids, max_new_tokens: int, trace_path: str) -> None:
    # 1. Determine activities (using cuda)
    activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]

    # 2. Run the function under the profiler
    with torch.profiler.profile(
        activities=activities,
        record_shapes=True
    ) as prof:
        _ = loop_fn(model, input_ids, max_new_tokens)

    # 3. Export the Chrome trace
    prof.export_chrome_trace(trace_path)

    # 4. Print the short table
    print(prof.key_averages().table(sort_by="self_cuda_time_total", row_limit=10))

## Part 2 — a fast, correct greedy decode loop

Return the same token ids as `baseline_loop` for the same fp32 model — shape
`(1, max_new_tokens)`. Start with what the baseline recomputes every step and
what it forces onto the host.

In [3]:
@torch.no_grad()
def optimized_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    # Compile the model directly with dynamic shape handling
    compiled_model = torch.compile(model, dynamic=True)

    current_ids = input_ids
    past_key_values = None
    out_tokens = []

    for i in range(max_new_tokens):
        current_ids = (current_ids if i == 0 else out_tokens[-1])

        # Run directly through compiled model
        out = compiled_model(input_ids=current_ids, past_key_values=past_key_values, use_cache=True)
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        past_key_values = out.past_key_values

        out_tokens.append(next_tok)

    return torch.cat(out_tokens, dim=1)

## Part 3 — build + time the optimized generation

Build the model however you like (dtype, `torch.compile`, CUDA graphs), run
`optimized_loop`, and return `(elapsed_seconds, generated_token_ids)`.

`elapsed_seconds` must be a warmed-up, timed measurement (use `time_loop`); it's
what gets compared to V0. Numerics-changing tricks (e.g. bf16) are fine here —
correctness is checked separately on the fp32 path.

In [4]:
def generate_optimized(max_new_tokens: int = MAX_NEW_TOKENS):
    """Return (elapsed_seconds, generated_token_ids) for your fastest setup."""

    # Allocate the model, in the same way and config as the baseline model.
    model, input_ids = build_model_and_input(torch.float32)

    # Generate the model output.
    gen_out_tokens = optimized_loop(model, input_ids, max_new_tokens)

    # Call the time loop.
    elapsed_seconds = time_loop(optimized_loop, model, input_ids, max_new_tokens)
    return elapsed_seconds, gen_out_tokens

## Self-check (small + fast, runs anywhere)

In [5]:
# SELF-CHECK — DO NOT EDIT.
_n = 24  # short, for speed
_model, _ids = build_model_and_input(torch.float32)

_ref = baseline_loop(_model, _ids, _n)
_cand = optimized_loop(_model, _ids, _n)
assert tuple(_cand.shape) == (1, _n), f"expected shape (1, {_n}), got {tuple(_cand.shape)}"
assert check_correctness(_ref, _cand), \
    "optimized_loop must reproduce the baseline greedy tokens EXACTLY (fp32)"
print("optimized_loop matches baseline   PASS")

_trace = os.path.join(RESULTS_DIR, "trace_check.json")
if os.path.exists(_trace):
    os.remove(_trace)
profile(baseline_loop, _model, _ids, 4, _trace)
assert os.path.exists(_trace) and os.path.getsize(_trace) > 0, \
    "profile() must write a non-empty Chrome trace file"
print("profile() writes a Chrome trace   PASS")
print()
print("All checks passed")

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


optimized_loop matches baseline   PASS


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         9.58%       1.863ms        14.49%       2.819ms      46.986us       2.133ms        58.83%       2.133ms      35.545us            60  
                                 ampere_sgemm_128x32_tn         0.00%       0.000us         0.00%       0.000us       0.000us     924.669us        25.51%     924.669us     231.167us             4  
         

## The full run — baseline vs optimized (GPU)

Prints your speedup and writes the two traces for the writeup.

In [7]:
# DO NOT EDIT — the full run.
N_NEW = MAX_NEW_TOKENS if torch.cuda.is_available() else 32
if N_NEW != MAX_NEW_TOKENS:
    print("CPU detected — trimmed to 32 new tokens as a smoke test. "
          "REPORTED NUMBERS MUST COME FROM A GPU RUN.")

# Baseline (V0), fp32 — also the correctness reference.
model_fp32, input_ids = build_model_and_input(torch.float32)
base_t = time_loop(baseline_loop, model_fp32, input_ids, N_NEW)
ref = baseline_loop(model_fp32, input_ids, N_NEW)
print(f"baseline V0: {base_t*1e3:.1f} ms/generation")

# Correctness of your loop on the SAME fp32 model.
cand = optimized_loop(model_fp32, input_ids, N_NEW)
ok = check_correctness(ref, cand)
print(f"optimized_loop correctness (fp32): {'PASS' if ok else 'FAIL'}")

# Optimized end-to-end timing (your choice of dtype/compile/graphs).
opt_t, _ = generate_optimized(N_NEW)
speedup = base_t / opt_t
print(f"optimized:   {opt_t*1e3:.1f} ms/generation")
print(f"speedup: {speedup:.2f}x  ->  {speedup_tier(speedup)}")
if not ok:
    print("WARNING: correctness FAILED — the speedup does not count.")

# Two traces for the writeup: baseline vs optimized.
profile(baseline_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_baseline.json"))
profile(optimized_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_optimized.json"))
print("wrote traces to results/hw2/ — open them at https://ui.perfetto.dev")

baseline V0: 496.7 ms/generation
optimized_loop correctness (fp32): PASS
optimized:   195.8 ms/generation
speedup: 2.54x  ->  >= 1.5x  (some speedup)
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm        12.00%      13.891ms        18.33%      21.207ms      44.182us      18.075ms        59.93%      18.075ms      37.656us           480  
                                 ampere_sgemm_128x32_tn  

---
## WRITEUP (be concrete and quantitative)

### Q1

From your **baseline trace**, what dominates the time? Name the specific symptom
you saw in the profiler (e.g. kernel-launch gaps, a long run of tiny kernels,
host syncs) and explain it.

**Your answer:**

The baseline is dominated by the attention - 60% of the CUDA time is dedicated to attention (aten::mm), and also the CPU highest percentage is dedicated to attention.

Other two high consumers of CUDA of the baseline are ampere_sgemm_128x32_tn and ampere_sgemm_32x32_sliced1x4_tn, which refer to matrix multiplications, which are also preformed massively during attention computation.

### Q2

List each optimization you applied and the speedup it contributed — measure them
**one at a time** (baseline → +KV cache → +compile → …). A small table is ideal.

**Your answer:**

> I have implemented 3 optimization phases: Prefill / KV Cache, avoid host syncs, and compilation, and here are the speedup obtained by each and by all:

```
Optimization  Speedup
KV Cache      1.25
Host Syncs    1.15
Compile       0.99
All           2.52
```

Strange enough: the compilation alone did not speedup performance at all, but without it the speedup degrades to 1.22.

### Q3

Which single change had the biggest impact, and why does it help THIS workload
(128 short decode steps on a tiny model) specifically?

**Your answer:**

> KV Cache had the biggest impact, when applied alone, but its advantage was only marginally, w.r.t. host syncs.
It is clear why KV cache improve performance: it saves a lot of computation.
However, due to the tiny model and short sequence, its advantage and conribution are relatively minor, as it requires also some overhead.

### Q4

Your decode loop is memory-bandwidth-bound (L1 §06, L2 §01). Which of your
optimizations attack memory traffic vs CPU/launch overhead? Which kind mattered
more here, and why?

**Your answer:**

> KV Cache and host syncs address memory traffic, while compilation attacks compute-bound limitations.

In this case we see clearly that handling memory-bound items is more effective than handling compute-bound.

This is probably due to the small size of the model, as we have seen already two cases that demonstrate that the impact of improving computation is proportional to the amount of comutation (e.g. the size of two multilied matrices). These two cases are: (1) arithmetic intensity of matrix multilpication (propotional to the dimension of the matrix) (2) compilation - its impact grows exponantially as the amount of computation grows.
And the size of the model induces exactly these two factors.